# 🚀 Data Augmentation para Facturas

Este notebook expande tu dataset de facturas generando múltiples variaciones mediante transformaciones geométricas.

**¿Qué hace?**
- Toma tus facturas originales (imagen + JSON)
- Genera 16 variaciones por cada factura (desplazamientos en diferentes direcciones)
- Multiplica tu dataset: 10 facturas → 170 facturas

**Resultado:** Dataset expandido listo para entrenar tu modelo de detección de campos.

## 📦 Paso 1: Instalación y Setup

In [ ]:
# Instalar dependencias
!pip install -q Pillow pdf2image numpy
!apt-get install -qq poppler-utils

print("✅ Dependencias instaladas")

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive montado")

In [ ]:
# Clonar el repositorio
!git clone https://github.com/GynoRomeroPrado/modificador-de-espacios-de-facturas.git
%cd modificador-de-espacios-de-facturas

print("✅ Repositorio clonado")

## ⚙️ Paso 2: Configuración

**IMPORTANTE:** Modifica estas rutas según tu estructura de Google Drive

In [ ]:
# ========================================
# CONFIGURA ESTAS RUTAS
# ========================================

# Directorio con tus facturas originales (imagen + JSON)
INPUT_DIR = "/content/drive/MyDrive/Facturas"

# Directorio donde guardar el dataset augmentado
OUTPUT_DIR = "/content/drive/MyDrive/Facturas_Procesadas"

# DPI para convertir PDFs a imágenes (mayor = mejor calidad, más pesado)
DPI = 200

print(f"📁 Input:  {INPUT_DIR}")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"🔧 DPI:    {DPI}")

## 🔍 Paso 3: Verificar Facturas de Entrada

In [ ]:
import os
from pathlib import Path

# Verificar que el directorio existe
if not os.path.exists(INPUT_DIR):
    print(f"❌ ERROR: El directorio {INPUT_DIR} no existe")
    print("\nPor favor:")
    print("1. Verifica que la ruta sea correcta")
    print("2. Asegúrate de haber montado Google Drive")
else:
    # Buscar archivos
    input_path = Path(INPUT_DIR)
    image_files = list(input_path.glob('*.pdf')) + list(input_path.glob('*.jpg')) + \
                  list(input_path.glob('*.jpeg')) + list(input_path.glob('*.png'))
    json_files = list(input_path.glob('*.json'))
    
    print(f"✅ Directorio encontrado: {INPUT_DIR}")
    print(f"\n📊 Contenido:")
    print(f"  • Archivos de imagen: {len(image_files)}")
    print(f"  • Archivos JSON:      {len(json_files)}")
    
    # Verificar pares
    pairs = 0
    for img_file in image_files:
        json_file = img_file.with_suffix('.json')
        if json_file.exists():
            pairs += 1
    
    print(f"  • Pares completos:    {pairs}")
    
    if pairs == 0:
        print("\n⚠️  ADVERTENCIA: No se encontraron pares completos (imagen + JSON)")
        print("   Cada factura debe tener su JSON correspondiente")
    else:
        print(f"\n✅ Listo para procesar {pairs} facturas")
        print(f"   Resultado esperado: {pairs} originales + {pairs * 16} augmentadas = {pairs * 17} facturas")

## 🚀 Paso 4: Procesar Dataset

Este paso genera todas las variaciones augmentadas.

In [ ]:
import sys
sys.path.append('/content/modificador-de-espacios-de-facturas/src')

from main import InvoiceDatasetAugmenter

# Crear augmenter
augmenter = InvoiceDatasetAugmenter(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    dpi=DPI
)

# Procesar dataset
stats = augmenter.process_dataset()

## ✅ Paso 5: Verificar Resultados

In [ ]:
import json

# Leer reporte
report_path = os.path.join(OUTPUT_DIR, 'dataset_report.json')

if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    
    print("📊 REPORTE DEL PROCESO")
    print("=" * 60)
    print(f"\n📅 Fecha: {report['timestamp']}")
    print(f"\n📁 Directorios:")
    print(f"  • Input:  {report['input_directory']}")
    print(f"  • Output: {report['output_directory']}")
    print(f"\n📊 Estadísticas:")
    print(f"  • Facturas originales:  {report['statistics']['original_invoices']}")
    print(f"  • Facturas augmentadas: {report['statistics']['augmented_invoices']}")
    print(f"  • Total de facturas:    {report['statistics']['total_invoices']}")
    
    if report['statistics']['errors']:
        print(f"\n⚠️  Errores: {len(report['statistics']['errors'])}")
        for error in report['statistics']['errors']:
            print(f"    - {error}")
    else:
        print("\n✅ Sin errores")
    
    print("\n" + "=" * 60)
else:
    print("❌ No se encontró el reporte")

## 👀 Paso 6: Explorar Resultados (Opcional)

In [ ]:
# Listar archivos generados
organized_dir = os.path.join(OUTPUT_DIR, 'organized')
augmented_dir = os.path.join(OUTPUT_DIR, 'augmented')

print("📁 ARCHIVOS GENERADOS\n")

if os.path.exists(organized_dir):
    organized_files = list(Path(organized_dir).glob('*'))
    print(f"📂 organized/ ({len(organized_files)} archivos)")
    for f in sorted(organized_files)[:10]:  # Mostrar solo los primeros 10
        print(f"  • {f.name}")
    if len(organized_files) > 10:
        print(f"  ... y {len(organized_files) - 10} más")

print()

if os.path.exists(augmented_dir):
    augmented_files = list(Path(augmented_dir).glob('*'))
    print(f"📂 augmented/ ({len(augmented_files)} archivos)")
    for f in sorted(augmented_files)[:10]:  # Mostrar solo los primeros 10
        print(f"  • {f.name}")
    if len(augmented_files) > 10:
        print(f"  ... y {len(augmented_files) - 10} más")

## 🖼️ Paso 7: Visualizar Ejemplo (Opcional)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Buscar primera factura augmentada
augmented_dir = os.path.join(OUTPUT_DIR, 'augmented')
augmented_images = sorted(Path(augmented_dir).glob('*.png'))[:4]

if augmented_images:
    fig, axes = plt.subplots(2, 2, figsize=(15, 15))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(augmented_images):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(img_path.name, fontsize=10)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    print("\n✅ Ejemplos de facturas augmentadas")
else:
    print("❌ No se encontraron imágenes augmentadas")

## 🎯 Próximos Pasos

1. ✅ Tu dataset ha sido expandido exitosamente
2. 📁 Los archivos están en tu Google Drive en: `{OUTPUT_DIR}`
3. 🚀 Ahora puedes usar este dataset expandido para:
   - Generar bounding boxes
   - Entrenar tu modelo de detección
   - Mejorar la precisión del sistema

**Estructura del output:**
```
Facturas_Procesadas/
├── organized/          # Facturas originales renombradas
│   ├── factura_0001.png
│   ├── factura_0001.json
│   └── ...
├── augmented/          # 16 variaciones por factura
│   ├── factura_0001_aug_01_derecha_small.png
│   ├── factura_0001_aug_01_derecha_small.json
│   └── ...
└── dataset_report.json # Reporte del proceso
```